# colab_11 — Geneformer continued pretraining (CPT), aggregated regime · N=1 pilot

**Regime #2/#3 of the FT strategy.** colab_09 froze the Geneformer *zero-shot* baseline; this notebook produces the first *adapted* checkpoint: **LoRA** continued-pretraining on the AD glia substrate with all studies concatenated + shuffled (the **aggregated** regime), one seed (**N=1 pilot**). CPT = *continued pretraining*: keep the model's original self-supervised objective (masked-gene reconstruction) but keep training it on our data — no external label, no classification head. This is the deliberate answer to the postdoc concern that "predictions suit a particular view": the model imposes no axis of its own.

**Scope is deliberately narrow.** Per `docs/EVALUATION_CONTRACT.md` the pilot's only job is to (1) run the CPT loop end-to-end and (2) confirm **detector #1** registers movement — median per-cell cosine similarity between the zero-shot and post-CPT embeddings drifts **> 0.05**. If detector #1 shows ca. 0 drift the run is inert and nothing downstream is worth interpreting. The full evals (#1 substate probe, #2 APOE recovery, #3 forgetting), detector #2, and the per-study regime are *separate later notebooks*, run only after this loop is validated and the provisional thresholds are refined against the pilot's own noise floor.

**Why Geneformer + aggregated first** (chosen over scGPT-first / per-study-first):

- *Geneformer before scGPT* — its CPT rides standard HuggingFace `Trainer` + `peft` LoRA tooling and ships an in-repo continual-learning precedent (the `Geneformer-V2-104M_CLcancer` checkpoint), so it is the lowest-risk path to a *correct* first CPT loop; scGPT's training is a bespoke loop from its own repo. Trade-off: Geneformer embeds ca. 29x slower than scGPT (colab_10b), so the full N=3 sweep is heavier — but getting the loop right, not iterating fast, is what the pilot is for.
- *Aggregated before per-study* — the pilot must be a single run to shake out the loop; aggregated is naturally one checkpoint on all cells, per-study is three. Aggregated also exercises the whole data pipeline and reads cleanest against the colab_09 baseline (same full substrate).

**Substrate + baseline it consumes:** the colab_07/08 labelled subsets (micro 54,805 / astro 87,783 = **142,588 cells, 145 donors**) and `geneformer/glia_geneformer_zeroshot.h5ad` from colab_09, realigned by `cell_index`.

**Open design points surfaced for the conceptual pass** (provisional values are set in-cell so the notebook runs; each is a decision, not a default): the donor-level split's disease-status stratifier (§3b — the subsets may not carry a diagnosis column), the LoRA configuration and masked-LM training budget (§5a — rank / target modules / mlm-probability / epochs / lr), and the masked-LM collator choice (§5a — a transparent custom collator vs. Geneformer's own `GeneformerPretrainer`).


## 1 — Setup

### 1a — Mount Drive, clone/pull the repo, install the Geneformer environment

Identical env to colab_09: Colab's **native** Python (Geneformer needs only >=3.10, no scvi-tools/scgpt), riding Colab's numpy-2 base, with `transformers` and `datasets` hard-pinned in `requirements_geneformer.txt`. CPT adds no new install — `peft` (LoRA) and `accelerate` (the `Trainer` backend) already come in through Geneformer's own `pip install .`. Requires a GPU runtime; training is heavier than colab_09's embed-only pass, so an **A100** is expected.


In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info >= (3, 10), (
    f"Geneformer needs Python >=3.10, got {sys.version_info[:2]}."
)

# Lean Geneformer-only stack: rides Colab's native numpy-2 base (see NUMPY POLICY in
# requirements_geneformer.txt); scGPT is NOT installed here. peft + accelerate arrive with
# Geneformer's own `pip install .` below -- CPT/LoRA needs no extra requirement.
!pip install -r {REPO_PATH}/requirements_geneformer.txt

GENEFORMER_REPO = "/content/Geneformer"
if not os.path.exists(GENEFORMER_REPO):
    !git lfs install
    !git clone https://huggingface.co/ctheodoris/Geneformer {GENEFORMER_REPO}
!cd {GENEFORMER_REPO} && pip install .

GENEFORMER_COMMIT = subprocess.run(
    ["git", "-C", GENEFORMER_REPO, "rev-parse", "HEAD"],
    capture_output=True, text=True).stdout.strip()
print("Python:", sys.version.split()[0], "| Geneformer commit:", GENEFORMER_COMMIT)

# Fail loud if no GPU: CPT on CPU is not viable at this scale.
import torch
assert torch.cuda.is_available(), "no CUDA GPU — select an A100 runtime before running CPT"
print("GPU:", torch.cuda.get_device_name(0))


## 2 — Environment capture

### 2a — pip freeze + env JSON (records the exact CPT-run stack)

Same freeze-then-document discipline as colab_09 §2a: snapshot the resolved versions + Geneformer commit + GPU/CUDA for the Methods record. This is a distinct run from colab_09, so it gets its own dated freeze file.


In [ ]:
import json, platform, subprocess, sys
from datetime import date

NOTEBOOK_ID = "colab_11"
TODAY = date.today().isoformat()
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

def _ver(mod):
    try:
        return __import__(mod).__version__
    except Exception:
        return None

env_snapshot = {
    "notebook_id":    NOTEBOOK_ID,
    "date":           TODAY,
    "python_version": sys.version,
    "platform":       platform.platform(),
    "os_release":     platform.release(),
    "gpu":            _run(["nvidia-smi", "-L"]),
    "cuda":           _run(["nvcc", "--version"]),
    "git_commit":     _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
    "geneformer_commit":    GENEFORMER_COMMIT,
    "scanpy_version":       _ver("scanpy"),
    "anndata_version":      _ver("anndata"),
    "torch_version":        _ver("torch"),
    "transformers_version": _ver("transformers"),
    "peft_version":         _ver("peft"),
    "accelerate_version":   _ver("accelerate"),
    "datasets_version":     _ver("datasets"),
    "geneformer_version":   _ver("geneformer"),
}
ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))


## 3 — Load the glia substrate + freeze the donor-level split

### 3a — Load both labelled subsets, concatenate, guard raw counts

Same load as colab_09 §3a: the colab_07/08 subsets share one gene panel (the full ca. 26,514-gene intersection, **raw counts** in `.X` — Geneformer rank-encodes raw counts, so log-normalized input would be silently wrong). Concatenate into one glia object, stamp a stable `cell_index` so every later artifact (tokenized dataset, post-CPT embedding) can be realigned to it, exactly as the zero-shot baseline was.


In [ ]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp

try:
    import psutil
    def _ram(tag):
        m = psutil.virtual_memory()
        print(f"[RAM] {tag:24s}: {m.used/1e9:5.1f} / {m.total/1e9:.1f} GB ({m.percent:.0f}%)")
except ImportError:
    def _ram(tag): pass

sc.settings.verbosity = 1

MICRO_PATH = os.path.join(DRIVE_ROOT, "micro_subset", "micro_subset.h5ad")
ASTRO_PATH = os.path.join(DRIVE_ROOT, "astro_subset", "astro_subset.h5ad")
for p in (MICRO_PATH, ASTRO_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"missing labelled subset {p} (colab_07 / colab_08 output)")

micro = sc.read_h5ad(MICRO_PATH)
astro = sc.read_h5ad(ASTRO_PATH)
print("microglia subset:", micro.shape)
print("astrocyte subset:", astro.shape)

assert list(micro.var_names) == list(astro.var_names), "gene panels differ between subsets"

micro.obs["lineage"] = "microglia"
astro.obs["lineage"] = "astrocyte"
# `diagnosis` is carried if present -- it is the disease-status stratifier for the §3b split.
KEEP_OBS = ["lineage", "substate", "apoe_carrier", "diagnosis", "study_id", "donor_id", "total_counts"]
micro.obs = micro.obs[[c for c in KEEP_OBS if c in micro.obs.columns]].copy()
astro.obs = astro.obs[[c for c in KEEP_OBS if c in astro.obs.columns]].copy()
glia = ad.concat([micro, astro], join="inner", index_unique="-")
del micro, astro; gc.collect()
glia.obs["cell_index"] = np.arange(glia.n_obs)
print("\ncombined glia:", glia.shape)
print("lineage:", glia.obs["lineage"].value_counts().to_dict())
print("apoe_carrier:", glia.obs["apoe_carrier"].value_counts(dropna=False).to_dict())
print("study_id:", glia.obs["study_id"].value_counts().to_dict())

# raw-counts guard -- Geneformer rank-encodes RAW counts; log-normalized input is silently wrong.
_idx = np.random.default_rng(0).choice(glia.n_obs, size=min(2000, glia.n_obs), replace=False)
Xs = glia.X[_idx]
data = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
frac_int = float(np.mean(np.mod(data, 1) == 0)) if data.size else 1.0
assert frac_int >= 0.99, f".X is not raw counts (int frac {frac_int:.3f}) -- FM input must be raw"
print("raw-counts int-frac:", round(frac_int, 3))
_ram("combined glia")


### 3b — Donor-level 70/15/15 split, stratified, frozen to disk

The **global held-out set** the whole eval contract rests on. Locked design: donor-level 70/15/15 (train/val/test), **stratified by disease x APOE x study**, computed **once** and reused identically by every regime and both FMs — so no cell from a test donor is ever trained on, and the split is not a researcher degree of freedom. This is the first notebook that needs it, so it is computed and **committed to the repo** (`outputs/donor_split.json`, keyed by `donor_id`) here; later notebooks load that file rather than recompute.

Split at the **donor** level (not cell level — cell-level splitting leaks a donor's identity across train/test). Fallback per the locked design if any `disease x APOE x study` stratum has < 3 donors: collapse APOE to "any-e4", else drop the thinnest study. The cell also audits the held-out study balance (no study > 60% of the test set, or the metric is a composition artifact).

*Open point for the conceptual pass:* the split needs a per-donor **disease status**. If `diagnosis` is absent from the subsets it must be joined from the upstream donor metadata (the ADNC / Braak table from colab_04/05) before this split is valid — the cell fails loud rather than silently stratifying on study x APOE alone.


In [ ]:
import json
from sklearn.model_selection import train_test_split

SPLIT_PATH   = os.path.join(REPO_PATH, "outputs", "donor_split.json")
SPLIT_SEED   = 0
SPLIT_FRACS  = {"train": 0.70, "val": 0.15, "test": 0.15}

# One row per donor with the stratification keys.
donor_tbl = (glia.obs[["donor_id", "study_id", "apoe_carrier", "diagnosis"]]
             if "diagnosis" in glia.obs.columns else None)
assert donor_tbl is not None, (
    "no `diagnosis` column on the glia subsets -- the locked split is stratified by "
    "disease x APOE x study. Join per-donor disease status from the upstream donor "
    "metadata (colab_04/05 ADNC/Braak table) before splitting."
)
donor_tbl = donor_tbl.drop_duplicates("donor_id").reset_index(drop=True)
# a donor must have a single (study, apoe, diagnosis) -- assert no donor spans two values.
assert donor_tbl["donor_id"].is_unique, "a donor_id maps to >1 (study/apoe/diagnosis) -- inspect before splitting"
print("donors:", len(donor_tbl), "| by study:", donor_tbl["study_id"].value_counts().to_dict())

strata = (donor_tbl["diagnosis"].astype(str) + " | "
          + donor_tbl["apoe_carrier"].astype(str) + " | "
          + donor_tbl["study_id"].astype(str))
small = strata.value_counts()[lambda s: s < 3]
if len(small):
    print("WARNING: strata with <3 donors (fallback = collapse APOE -> any-e4):\n", small.to_dict())
    apoe_collapsed = donor_tbl["apoe_carrier"].astype(str).replace({"noncarrier": "any", "carrier": "any"})
    # collapse APOE only inside offending strata is fiddly; per locked design collapse APOE globally.
    strata = (donor_tbl["diagnosis"].astype(str) + " | any | " + donor_tbl["study_id"].astype(str))

# 70 / 15 / 15 at donor level, stratified. Two-step: first hold out test, then split val off train.
d_train, d_test = train_test_split(
    donor_tbl["donor_id"], test_size=SPLIT_FRACS["test"], random_state=SPLIT_SEED, stratify=strata)
strata_tr = strata.loc[donor_tbl["donor_id"].isin(d_train).values]
d_train, d_val = train_test_split(
    d_train, test_size=SPLIT_FRACS["val"] / (SPLIT_FRACS["train"] + SPLIT_FRACS["val"]),
    random_state=SPLIT_SEED, stratify=strata_tr)

split_map = {d: "train" for d in d_train} | {d: "val" for d in d_val} | {d: "test" for d in d_test}
glia.obs["split"] = glia.obs["donor_id"].map(split_map).astype("category")

# Audits: cell counts per split, held-out study balance (no study > 60% of test).
print("\ncells per split:", glia.obs["split"].value_counts().to_dict())
test_by_study = glia.obs.loc[glia.obs["split"] == "test", "study_id"].value_counts(normalize=True)
print("test-set study fractions:", (test_by_study.round(3)).to_dict())
assert (test_by_study <= 0.60).all(), "a study is >60% of the test set -- held-out is composition-imbalanced"

# Freeze to the repo (committed): donor -> split, plus the recipe for reproducibility.
split_artifact = {
    "seed": SPLIT_SEED, "fracs": SPLIT_FRACS, "level": "donor",
    "stratify": "diagnosis x apoe_carrier x study_id",
    "n_donors": {k: int(sum(v == k for v in split_map.values())) for k in ("train", "val", "test")},
    "donor_split": split_map,
}
with open(SPLIT_PATH, "w") as f:
    json.dump(split_artifact, f, indent=2)
print("froze donor split ->", SPLIT_PATH)


## 4 — Tokenize the substrate

### 4a — Tokenize the full glia object into a Geneformer dataset

Tokenize **all** cells once (like colab_09 §5a), carrying `split` and the labels through into the `.dataset`. §5 then trains on the `split == "train"` rows only, and §6 embeds every row — one tokenization serves both, and guarantees the CPT-input encoding is bit-identical to the zero-shot baseline's. Geneformer normalizes each cell by its total count, so recompute `n_counts` from raw `.X` and fail loud on any zero-count cell.


In [ ]:
from geneformer import TranscriptomeTokenizer

glia.obs["n_counts"] = np.asarray(glia.X.sum(axis=1)).ravel()
assert (glia.obs["n_counts"] > 0).all(), "cells with zero counts present -- should have been QC'd upstream"

TOK_IN_DIR  = "/content/gf_token_in"
TOK_OUT_DIR = "/content/gf_token_out"
os.makedirs(TOK_IN_DIR, exist_ok=True)
os.makedirs(TOK_OUT_DIR, exist_ok=True)
glia.write_h5ad(os.path.join(TOK_IN_DIR, "glia.h5ad"))

# split + labels carried into every tokenized cell; cell_index is the realignment key.
CUSTOM_ATTRS = {
    "cell_index":   "cell_index",
    "split":        "split",
    "lineage":      "lineage",
    "substate":     "substate",
    "apoe_carrier": "apoe_carrier",
    "study_id":     "study_id",
    "donor_id":     "donor_id",
}
# Same defaults as colab_09 (V2, input size 4096, GC104M dictionaries) so encoding matches the baseline.
tk = TranscriptomeTokenizer(CUSTOM_ATTRS, nproc=4)
tk.tokenize_data(TOK_IN_DIR, TOK_OUT_DIR, "glia_cpt", file_format="h5ad")
TOKENIZED_DATASET = os.path.join(TOK_OUT_DIR, "glia_cpt.dataset")
print("tokenized ->", TOKENIZED_DATASET)


## 5 — Continued pretraining with LoRA

### 5a — Load the pretrained checkpoint, attach a LoRA adapter, configure the masked-LM trainer

Warm-start from the **same** `Geneformer-V2-104M` checkpoint colab_09 embedded, as a `BertForMaskedLM` (the masked-gene reconstruction head — CPT's training signal, no external label). Freeze the base weights and train only a **LoRA** adapter (`peft`), consistent across every regime for a fair comparison and to fit Colab compute.

The values below are **provisional pilot settings**, set so the loop runs — each is a decision to ratify, not a default:

- **LoRA:** `r=8`, `alpha=16`, `dropout=0.05`, `target_modules=["query","key","value"]` (the attention projections). Whether to also adapt the output `dense`/intermediate layers or leave the MLM head trainable (`modules_to_save`) is open.
- **Masking:** `mlm_probability=0.15` (BERT default) using Geneformer's mask/pad token ids from `TOKEN_DICTIONARY_FILE`. Whether to reuse Geneformer's own `GeneformerPretrainer` collator instead of the transparent custom collator here is open — the custom one is written for auditability on a single GPU.
- **Budget:** provisional `num_train_epochs=1`, `lr=5e-4` (LoRA tolerates a higher LR than full-FT), `warmup_ratio=0.05`, batch tuned to A100 memory. The pilot exists to reveal the real budget from detector #1's drift.


In [ ]:
import pickle, torch
from datasets import load_from_disk
from transformers import BertForMaskedLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from geneformer import TOKEN_DICTIONARY_FILE

# --- provisional pilot hyperparameters (ratify before N=3) ---
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05
LORA_TARGETS   = ["query", "key", "value"]
MLM_PROB       = 0.15
NUM_EPOCHS     = 1
LEARNING_RATE  = 5e-4
PER_DEV_BATCH  = 8
WARMUP_RATIO   = 0.05
SEED           = 0

with open(TOKEN_DICTIONARY_FILE, "rb") as f:
    token_dict = pickle.load(f)
PAD_ID  = token_dict.get("<pad>")
MASK_ID = token_dict.get("<mask>")
assert PAD_ID is not None and MASK_ID is not None, "pad/mask token id missing from token dictionary"
VOCAB_SIZE = len(token_dict)

MODEL_DIR = os.path.join(GENEFORMER_REPO, "Geneformer-V2-104M")
assert os.path.exists(MODEL_DIR), f"model dir not found: {MODEL_DIR}"
base_model = BertForMaskedLM.from_pretrained(MODEL_DIR)

lora_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS, bias="none", task_type=TaskType.FEATURE_EXTRACTION,
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

# Train only on the held-in donors; the tokenized dataset carries `split`.
full_ds  = load_from_disk(TOKENIZED_DATASET)
train_ds = full_ds.filter(lambda ex: ex["split"] == "train")
print("train cells (split==train):", len(train_ds), "of", len(full_ds))

# Transparent masked-LM collator over Geneformer token ids: pad to batch max, mask MLM_PROB of
# real (non-pad) tokens, standard 80/10/10 mask/random/keep, labels = -100 on unmasked positions.
# NOTE: exact special-token handling to be confirmed against Geneformer's pretraining collator.
class MlmCollator:
    def __init__(self, pad_id, mask_id, vocab_size, mlm_prob):
        self.pad_id, self.mask_id, self.vocab_size, self.mlm_prob = pad_id, mask_id, vocab_size, mlm_prob
    def __call__(self, examples):
        seqs = [ex["input_ids"] for ex in examples]
        maxlen = max(len(s) for s in seqs)
        input_ids = torch.full((len(seqs), maxlen), self.pad_id, dtype=torch.long)
        attn = torch.zeros((len(seqs), maxlen), dtype=torch.long)
        for i, s in enumerate(seqs):
            input_ids[i, :len(s)] = torch.tensor(s, dtype=torch.long); attn[i, :len(s)] = 1
        labels = input_ids.clone()
        probs = torch.full(input_ids.shape, self.mlm_prob)
        probs[input_ids == self.pad_id] = 0.0
        masked = torch.bernoulli(probs).bool()
        labels[~masked] = -100
        r = torch.rand(input_ids.shape)
        input_ids[masked & (r < 0.8)] = self.mask_id
        rand_tok = torch.randint(self.vocab_size, input_ids.shape, dtype=torch.long)
        replace = masked & (r >= 0.8) & (r < 0.9)
        input_ids[replace] = rand_tok[replace]
        return {"input_ids": input_ids, "attention_mask": attn, "labels": labels}

collator = MlmCollator(PAD_ID, MASK_ID, VOCAB_SIZE, MLM_PROB)

OUT_DIR = "/content/gf_cpt_out"
targs = TrainingArguments(
    output_dir=OUT_DIR, overwrite_output_dir=True,
    num_train_epochs=NUM_EPOCHS, per_device_train_batch_size=PER_DEV_BATCH,
    learning_rate=LEARNING_RATE, warmup_ratio=WARMUP_RATIO,
    logging_steps=50, save_strategy="no", report_to="none",
    bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
    seed=SEED, dataloader_num_workers=2,
)
trainer = Trainer(model=model, args=targs, train_dataset=train_ds, data_collator=collator)
print("trainer ready | epochs", NUM_EPOCHS, "| lr", LEARNING_RATE, "| batch", PER_DEV_BATCH)


### 5b — Run CPT and save the LoRA adapter

Train, then save the **adapter only** (ca. hundreds of MB, per the storage plan — never the full merged weights) to Drive. The training loss curve is the first sanity signal that masked-gene reconstruction is actually happening; detector #1 in §7 is the formal gate.


In [ ]:
train_result = trainer.train()
print("train loss:", round(train_result.training_loss, 4))

ADAPTER_DIR = os.path.join(DRIVE_ROOT, "geneformer", "cpt_aggregated_seed0_adapter")
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
print("saved LoRA adapter ->", ADAPTER_DIR)


## 6 — Re-embed the full substrate from the adapted model

### 6a — Merge the adapter and extract post-CPT cell embeddings

Merge the LoRA adapter into the base weights to produce a standalone adapted checkpoint on disk, then run the **same** `EmbExtractor` pass colab_09 used (cell embedding, final layer, all cells) over the full tokenized dataset. Realign by `cell_index` so the post-CPT vectors line up cell-for-cell with the zero-shot baseline for detector #1. The merged checkpoint is written to a temporary dir (not Drive — only the compact adapter is kept long-term).


In [ ]:
from geneformer import EmbExtractor

MERGED_DIR = "/content/gf_cpt_merged"
merged = model.merge_and_unload()          # fold LoRA into the base weights
merged.save_pretrained(MERGED_DIR)
# EmbExtractor loads a checkpoint dir by path; it needs the tokenizer/config files too.
base_model.config.to_json_file(os.path.join(MERGED_DIR, "config.json"))
print("merged adapted checkpoint ->", MERGED_DIR)

ee = EmbExtractor(
    model_type="Pretrained", num_classes=0, emb_mode="cell",
    max_ncells=None, emb_layer=-1,
    emb_label=["cell_index", "split", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"],
    forward_batch_size=64, nproc=4,
)
EMB_OUT_DIR = "/content/gf_cpt_emb_out"
os.makedirs(EMB_OUT_DIR, exist_ok=True)
emb_df = ee.extract_embs(MERGED_DIR, TOKENIZED_DATASET, EMB_OUT_DIR, "glia_cpt")
print("embedding frame:", emb_df.shape)

label_cols = ["cell_index", "split", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]
emb_cols = [c for c in emb_df.columns if c not in label_cols]
emb_df = emb_df.set_index("cell_index").reindex(glia.obs["cell_index"].values)
assert emb_df[emb_cols].notna().all().all(), "embedding rows missing after realignment to cell_index"
X_cpt = emb_df[emb_cols].to_numpy(dtype=np.float32)
glia.obsm["X_geneformer_cpt"] = X_cpt
print("X_geneformer_cpt:", X_cpt.shape, "| emb dim:", X_cpt.shape[1])


## 7 — Detector #1: is the embedding changing?

### 7a — Median per-cell cosine, zero-shot -> post-CPT, against the > 0.05 gate

The pilot's formal gate (`docs/EVALUATION_CONTRACT.md`, detector #1). Load the frozen colab_09 zero-shot baseline, align it to this run by `cell_index`, and compute per-cell cosine similarity between each cell's zero-shot and post-CPT vector. **Drift = 1 - median cosine**; **> 0.05 = registered** (CPT had a measurable effect). Reported overall and on the **test split** specifically (the held-out donors the evals will actually score). A near-zero drift means the run is inert — the loop or the training budget needs fixing before anything downstream is interpretable.


In [ ]:
ZEROSHOT_PATH = os.path.join(DRIVE_ROOT, "geneformer", "glia_geneformer_zeroshot.h5ad")
assert os.path.exists(ZEROSHOT_PATH), f"missing colab_09 baseline {ZEROSHOT_PATH}"
zs = sc.read_h5ad(ZEROSHOT_PATH)

# align baseline to this run by cell_index
zs_df = pd.DataFrame(zs.X, index=zs.obs["cell_index"].values)
zs_aligned = zs_df.reindex(glia.obs["cell_index"].values)
assert zs_aligned.notna().all().all(), "baseline rows missing after cell_index alignment"
X_zs = zs_aligned.to_numpy(dtype=np.float32)
assert X_zs.shape == glia.obsm["X_geneformer_cpt"].shape, "zero-shot / post-CPT embedding dims differ"

def _per_cell_cosine(A, B):
    num = (A * B).sum(1)
    den = np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1) + 1e-12
    return num / den

cos = _per_cell_cosine(X_zs, glia.obsm["X_geneformer_cpt"])
median_cos = float(np.median(cos))
drift = 1.0 - median_cos
registered = drift > 0.05

test_mask = (glia.obs["split"] == "test").values
drift_test = 1.0 - float(np.median(cos[test_mask])) if test_mask.any() else float("nan")

print(f"median cosine (all):  {median_cos:.4f}  -> drift {drift:.4f}")
print(f"median cosine (test): {1.0 - drift_test:.4f} -> drift {drift_test:.4f}")
print(f"detector #1 (>0.05): {'REGISTERED' if registered else 'INERT -- run is not usable, fix before interpreting'}")

DETECTOR1 = {
    "median_cosine_all": median_cos, "drift_all": drift,
    "drift_test": drift_test, "threshold": 0.05, "registered": bool(registered),
}


## 8 — Save + handoff

### 8a — Save the post-CPT embedding, append the audit trace, print commit commands

Persist the post-CPT embedding as a lean labelled artifact (vectors + labels + `split`, no gene matrix — mirrors the zero-shot baseline so the eval notebooks can diff them by `cell_index`). Append this run under `geneformer_cpt_aggregated` in the cumulative `outputs/audit_report.json`, recording the adapter path, LoRA config, training budget, and the detector #1 verdict. Git artifacts to commit: the freeze/env JSON, the frozen donor split, and the audit report. The **adapter** and **embedding** stay on Drive (too large for git; figures/large binaries are gitignored by design).


In [ ]:
import shlex

GF_DIR = os.path.join(DRIVE_ROOT, "geneformer")
emb_adata = ad.AnnData(
    X=glia.obsm["X_geneformer_cpt"],
    obs=glia.obs[["cell_index", "split", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]].copy(),
)
EMB_PATH = os.path.join(GF_DIR, "glia_geneformer_cpt_aggregated_seed0.h5ad")
emb_adata.write_h5ad(EMB_PATH)
print("saved post-CPT embedding ->", EMB_PATH, f"({os.path.getsize(EMB_PATH)/1e9:.2f} GB)")

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)
report["geneformer_cpt_aggregated"] = {
    "status": "computed", "date": TODAY, "regime": "aggregated", "seed": SEED,
    "model_dir": os.path.basename(MODEL_DIR), "geneformer_commit": GENEFORMER_COMMIT,
    "n_cells": int(glia.n_obs), "n_train_cells": int((glia.obs["split"] == "train").sum()),
    "emb_dim": int(glia.obsm["X_geneformer_cpt"].shape[1]),
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "targets": LORA_TARGETS},
    "train": {"epochs": NUM_EPOCHS, "lr": LEARNING_RATE, "batch": PER_DEV_BATCH,
              "mlm_prob": MLM_PROB, "train_loss": round(float(train_result.training_loss), 4)},
    "detector_1": DETECTOR1,
    "adapter_file": os.path.relpath(ADAPTER_DIR, DRIVE_ROOT),
    "embedding_file": os.path.relpath(EMB_PATH, DRIVE_ROOT),
    "donor_split_file": os.path.relpath(SPLIT_PATH, REPO_PATH),
}
with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)
print("audit trace appended ->", AUDIT_REPORT_PATH)

rel = [os.path.relpath(p, REPO_PATH) for p in (FREEZE_PATH, ENV_JSON_PATH, SPLIT_PATH, AUDIT_REPORT_PATH)]
print("\n=== Commit + push (from WSL -- Colab has no git creds) ===")
print("  git add " + " ".join(shlex.quote(r) for r in rel))
print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git commit -m "
      "'colab_11: Geneformer CPT (aggregated, N=1 pilot) + donor split + detector #1'")
print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git push")


### Carried forward

- **This notebook produces:** the frozen donor-level split (`outputs/donor_split.json`, the shared held-out for every regime/FM), one aggregated-regime LoRA adapter + post-CPT embedding on Drive, and the detector #1 verdict for the pilot.
- **Next, once detector #1 registers:** refine the provisional eval-contract thresholds against this pilot's noise floor (the one permitted threshold edit), then the per-study regime, detector #2 (forgetting), and evals #1/#2/#3 — and the scGPT CPT parallel.
- **Open items to settle in the conceptual pass:** the disease-status stratifier source for §3b; the LoRA config + training budget in §5a; custom collator vs. `GeneformerPretrainer`; whether detector #1 should be measured on all cells or held-out only.
